# Lecture 03: Training Models and Measuring Them

This notebook continues from `01_EDA_Feature_Engineering.ipynb`. It starts from the cleaned data, without the coded late-payment rows.

For Model A you compare three models:

1. **Your rule-based baseline** from lecture 2.
2. **Logistic regression.**
3. **A random forest.**

Two rules hold throughout. All comparisons use the validation set. The test set stays untouched, because it is used once, at the very end of the project. Every random step has a fixed seed, so that you get the same numbers on every run.

In [ ]:
!uv pip install pandas matplotlib scikit-learn

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

BLUE, PINK, SLATE, GREY = "#77C5D8", "#E5007D", "#466873", "#9A9A9A"
SEED = 42

df = pd.read_csv("../../data/credit_risk_dataset.csv").drop_duplicates()

# Drop the rows with the codes 96 and 98 in the late-payment columns, as in notebook 01.
late_cols = ["late_30_59", "late_60_89", "late_90"]
df = df[~df[late_cols].isin([96, 98]).any(axis=1)]

X, y = df.drop(columns="dlq_2yrs"), df["dlq_2yrs"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
print(len(X_train), len(X_val), len(X_test))

## 1. The baseline

A trained model only earns its place if it beats a baseline. Your main baseline is the rule-based model you built for Model A in lecture 2.

The simplest possible baseline always predicts the most frequent class. Any model must beat this one by a wide margin.

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"validation accuracy of the dummy baseline: {dummy.score(X_val, y_val):.3f}")

## 2. The generated loan book from the slides

For simplicity we use a small generated loan book with two features, `rev_util` and `age`. With two features you can draw what a model learns. The cell below makes the same 300 applicants as the slides, so you can reproduce their numbers and change them.

The risk of default rises with `rev_util` and falls with `age`. Chance decides the rest. Half of the applicants form the training set, the other half the validation set.

About half of the applicants default, so accuracy is a fair measure on this loan book.

In [ ]:
def make_loan_book(n=300, seed=17):
    """
    Create a generated loan book with two features and a binary target. The probability of default rises with rev_util and falls with age.

    :param n: Number of applicants to generate
    :param seed: Random seed for reproducibility
    :return: DataFrame with columns 'rev_util', 'age', and 'default' (0 or 1)
    """
    rng = np.random.default_rng(seed)
    age = rng.uniform(20, 75, n).round(0)
    rev_util = rng.beta(1.6, 1.6, n).round(2)
    z = 2.5 + 7.0 * rev_util - 0.13 * age
    default = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
    return pd.DataFrame({"rev_util": rev_util, "age": age, "default": default})

TOY = ["rev_util", "age"]
book = make_loan_book()
toy_train, toy_val = train_test_split(book, test_size=0.5, stratify=book["default"], random_state=SEED)
new_applicant = pd.DataFrame({"rev_util": [0.70], "age": [45.0]})

ax = toy_train.plot.scatter(x="rev_util", y="age", c=toy_train["default"].map({0: BLUE, 1: PINK}), figsize=(5, 3.6))
ax.scatter(new_applicant.rev_util, new_applicant.age, marker="*", s=200, color="black")
ax.set_title("pink: defaulted, blue: repaid, star: new applicant")
plt.show()

### Logistic regression on two features

Without scaling and without regularisation (`C=np.inf`), each weight reads in the units of its feature. The new applicant uses 70% of the credit limit and is 45 years old.

In [ ]:
from sklearn.linear_model import LogisticRegression

toy_logreg = LogisticRegression(C=np.inf, max_iter=5000).fit(toy_train[TOY], toy_train["default"])
b = toy_logreg.intercept_[0]
w_rev, w_age = toy_logreg.coef_[0]
z = b + w_rev * 0.70 + w_age * 45
print(f"z = {b:.2f} + {w_rev:.2f} * 0.70 + ({w_age:.3f}) * 45 = {z:.2f}")
print(f"predicted probability of default: {1 / (1 + np.exp(-z)):.2f}")

### A decision tree on two features

The first question splits on `rev_util`, the next ones on `age`. Then compare a tree of depth 2 with a tree of depth 12. The deep tree is almost perfect on the training half and worse on the validation half. This is the overfitting from the slides.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

for depth in [2, 12]:
    toy_tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(toy_train[TOY], toy_train["default"])
    if depth == 2:
        print(export_text(toy_tree, feature_names=TOY))
        print(f"the new applicant: p = {toy_tree.predict_proba(new_applicant)[0, 1]:.2f}")
    print(f"depth {depth}: training accuracy {toy_tree.score(toy_train[TOY], toy_train['default']):.2f}, "
          f"validation accuracy {toy_tree.score(toy_val[TOY], toy_val['default']):.2f}")

## 3. Logistic regression on the project data

Logistic regression gives every feature one learned weight. It computes a weighted sum, $z = \sum_j w_j x_j + b$, and the sigmoid turns the sum into a probability, $\hat{p} = \sigma(z) = \frac{1}{1 + e^{-z}}$.

The scaler and the model sit in one `Pipeline`. The pipeline fits the scaler on the training data only, so it cannot leak validation data into training.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train)
print(f"validation accuracy: {logreg.score(X_val, y_val):.3f}")

Each weight shows how one feature moves the score. After scaling, the weights are comparable across features. A positive weight raises the predicted probability of default.

In [ ]:
weights = pd.Series(logreg[-1].coef_[0], index=X.columns).sort_values()
ax = weights.plot.barh(color=[PINK if w > 0 else BLUE for w in weights], figsize=(6, 4))
ax.set_xlabel("weight after scaling")
ax.set_title("Logistic regression weights")
plt.tight_layout()
plt.show()

### One applicant, step by step

The slides worked through the new applicant of the made-up loan book. Here is the same calculation for one real applicant of the validation set. Each feature contributes its weight times its scaled value. The contributions plus the bias give the score z. The sigmoid turns z into the predicted probability.

In [ ]:
scaler, model = logreg[0], logreg[-1]
p_val = logreg.predict_proba(X_val)[:, 1]

candidates = X_val[(y_val == 1).values & (p_val > 0.75) & (p_val < 0.85)
                   & (X_val["late_90"] <= 1).values & (X_val["rev_util"] < 1.2).values]
applicant = candidates.iloc[[0]]
print(applicant.iloc[0].to_dict())

contributions = pd.Series(scaler.transform(applicant)[0] * model.coef_[0], index=X.columns)
contributions = contributions.reindex(contributions.abs().sort_values(ascending=False).index)
z = contributions.sum() + model.intercept_[0]
print(f"bias b = {model.intercept_[0]:+.2f}")
print(contributions.round(2).to_string())
print(f"z = {z:.2f}, predicted probability = {1 / (1 + np.exp(-z)):.2f}")

### Optional: the training loop by hand

This is the loop from the slide in a few lines of NumPy. Each epoch predicts, computes the log-loss, computes the gradient and updates the weights.

$$L = -\frac{1}{n} \sum_i \left[ y_i \log \hat{p}_i + (1 - y_i) \log(1 - \hat{p}_i) \right] \qquad w \leftarrow w - \eta \frac{\partial L}{\partial w}$$

In [ ]:
scaler = StandardScaler().fit(X_train)
Xs_train, Xs_val = scaler.transform(X_train), scaler.transform(X_val)
yt, yv = y_train.values, y_val.values

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(y_true, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

w, b, lr = np.zeros(Xs_train.shape[1]), 0.0, 0.5
history = []
for epoch in range(300):
    p = sigmoid(Xs_train @ w + b)
    w -= lr * Xs_train.T @ (p - yt) / len(yt)
    b -= lr * np.mean(p - yt)
    history.append((log_loss(yt, p), log_loss(yv, sigmoid(Xs_val @ w + b))))

history = np.array(history)
plt.plot(history[:, 0], color=BLUE, label="training loss")
plt.plot(history[:, 1], color=PINK, ls="--", label="validation loss")
plt.xlabel("epoch"); plt.ylabel("log-loss"); plt.legend(frameon=False)
plt.show()

Try a learning rate of 0.05 and of 5. What happens to the curve in each case?

## 4. Decision trees and overfitting

A decision tree asks yes or no questions about the features. Print a small tree and read it. Each leaf shows how many training applicants end there, by class.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

small_tree = DecisionTreeClassifier(max_depth=2, random_state=SEED).fit(X_train, y_train)
print(export_text(small_tree, feature_names=list(X.columns), show_weights=True))

A deeper tree fits the training data more closely. Watch where the validation accuracy stops improving.

In [ ]:
depths = range(1, 21)
train_acc, val_acc = [], []
for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=SEED).fit(X_train, y_train)
    train_acc.append(tree.score(X_train, y_train))
    val_acc.append(tree.score(X_val, y_val))

plt.plot(depths, train_acc, color=BLUE, marker="o", label="training")
plt.plot(depths, val_acc, color=PINK, marker="o", label="validation")
plt.xlabel("maximum tree depth"); plt.ylabel("accuracy"); plt.legend(frameon=False)
plt.show()
print(f"best validation depth: {int(np.argmax(val_acc)) + 1}")

## 5. Randomness

The split is random. Change only its seed and train the same logistic regression again. The spread you see is the reason why every run must record its seeds.

In [ ]:
scores = []
for split_seed in range(10):
    Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=split_seed)
    Xv, _, yv_, _ = train_test_split(Xtmp, ytmp, test_size=0.50, stratify=ytmp, random_state=split_seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
    scores.append(model.score(Xv, yv_))
print(f"ten split seeds: validation accuracy from {min(scores):.3f} to {max(scores):.3f}")

## 6. Classification metrics

Every metric starts from the confusion matrix. The table below uses a threshold of 0.5 on the predicted probability.


| Metric | Formula |
|---|---|
| Accuracy | $\frac{TP + TN}{TP + TN + FP + FN}$ |
| Precision | $\frac{TP}{TP + FP}$ |
| Recall | $\frac{TP}{TP + FN}$ |
| Specificity | $\frac{TN}{TN + FP}$ |
| F1 score | $\frac{2 \cdot P \cdot R}{P + R}$ |
| MCC | $\frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP + FP)(TP + FN)(TN + FP)(TN + FN)}}$ |

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score, matthews_corrcoef,
                             precision_score, recall_score, roc_auc_score)

def metrics(model, X_eval, y_eval, threshold=0.5):
    p = model.predict_proba(X_eval)[:, 1]
    pred = (p >= threshold).astype(int)
    tp, fn, fp, tn = confusion_matrix(y_eval, pred, labels=[1, 0]).ravel()   # TP top left
    return {"TP": tp, "FN": fn, "FP": fp, "TN": tn,
            "accuracy": accuracy_score(y_eval, pred),
            "precision": precision_score(y_eval, pred, zero_division=0),
            "recall": recall_score(y_eval, pred),
            "specificity": tn / (tn + fp),
            "F1": f1_score(y_eval, pred, zero_division=0),
            "MCC": matthews_corrcoef(y_eval, pred),
            "ROC AUC": roc_auc_score(y_eval, p)}

pd.DataFrame({"dummy": metrics(dummy, X_val, y_val),
              "logistic regression": metrics(logreg, X_val, y_val)}).round(3)

The confusion matrix of logistic regression, with the actual class in the rows and the predicted class in the columns, and the report that `classification_report` prints from the same decisions:

In [ ]:
pred_val = (logreg.predict_proba(X_val)[:, 1] >= 0.5).astype(int)
names = ["default", "no default"]
print(pd.DataFrame(confusion_matrix(y_val, pred_val, labels=[1, 0]),
                   index=[f"actual {n}" for n in names], columns=[f"predicted {n}" for n in names]))
print()
print(classification_report(y_val, pred_val, target_names=["no default", "default"]))

## 7. ROC and precision-recall curves

The ROC curve plots the true positive rate against the false positive rate for every threshold. AUC is the area under it. The precision-recall curve shows the trade-off you face when you move the threshold.

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_curve

p = logreg.predict_proba(X_val)[:, 1]
fpr, tpr, _ = roc_curve(y_val, p)
prec, rec, _ = precision_recall_curve(y_val, p)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(fpr, tpr, color=BLUE, label=f"logistic regression (AUC = {roc_auc_score(y_val, p):.2f})")
axes[0].plot([0, 1], [0, 1], color=GREY, ls="--", label="random guessing")
axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("true positive rate"); axes[0].set_title("ROC curve")
axes[1].plot(rec, prec, color=BLUE, label=f"logistic regression (AP = {average_precision_score(y_val, p):.2f})")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].set_title("Precision-recall curve")
for ax in axes:
    ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

## 8. When defaults are rare: the imbalanced loan book

The project data is balanced. Real loan books are not. The next cells generate the imbalanced loan book from the slides, where only about 5% of applicants default. It is generated data, so the numbers say nothing about our project. They show how metrics behave when one class is rare.

In [ ]:
from sklearn.datasets import make_classification

Xg, yg = make_classification(n_samples=20000, n_features=8, n_informative=4, n_redundant=2,
                             weights=[0.95, 0.05], class_sep=0.8, flip_y=0.01, random_state=7)
Xg_train, Xg_test, yg_train, yg_test = train_test_split(Xg, yg, test_size=0.30, stratify=yg, random_state=SEED)
print(f"default rate: {yg.mean():.1%}")

g_dummy = DummyClassifier(strategy="most_frequent").fit(Xg_train, yg_train)
g_logreg = LogisticRegression(max_iter=1000).fit(Xg_train, yg_train)

pd.DataFrame({"always no default": metrics(g_dummy, Xg_test, yg_test),
              "logistic regression": metrics(g_logreg, Xg_test, yg_test)}).round(3)

Always predicting "no default" reaches about 94.5% accuracy and catches no default at all. Look at recall and F1 instead. Then look at the precision-recall curve. Its random baseline is the default rate, not 0.5.

### Choosing the threshold by cost

If you can price each error, you can compute the cost of every threshold:

$$\text{Cost}(t) = c_{FN} \cdot FN(t) + c_{FP} \cdot FP(t)$$

The cell assumes that a missed default costs ten times as much as a rejected good applicant. That ratio is an assumption for the example. Change it and watch the best threshold move.

In [ ]:
COST_FN, COST_FP = 10, 1
p = g_logreg.predict_proba(Xg_test)[:, 1]
thresholds = np.linspace(0.01, 0.99, 99)
costs = []
for t in thresholds:
    tp, fn, fp, tn = confusion_matrix(yg_test, (p >= t).astype(int), labels=[1, 0]).ravel()
    costs.append(COST_FN * fn + COST_FP * fp)

best = thresholds[int(np.argmin(costs))]
for t in [best, 0.2, 0.5]:
    tp, fn, fp, tn = confusion_matrix(yg_test, (p >= t).astype(int), labels=[1, 0]).ravel()
    print(f"threshold {t:.2f}: {fn} missed defaults, {fp} rejected good applicants, cost {COST_FN * fn + COST_FP * fp}")

plt.plot(thresholds, costs, color=SLATE)
plt.axvline(best, color=PINK, ls="--", label=f"lowest cost at {best:.2f}")
plt.axvline(0.5, color=GREY, ls=":", label="default 0.5")
plt.xlabel("decision threshold"); plt.ylabel("total cost"); plt.legend(frameon=False)
plt.show()

## 9. Report mean and standard deviation

One validation score is one draw. Another split or seed gives another number. Five-fold cross-validation trains five times, each time on four fifths of the training data, and scores the fifth. Report the mean and the standard deviation of the five scores:

$$\bar{s} = \frac{1}{k} \sum_i s_i \qquad \sigma = \sqrt{\frac{1}{k - 1} \sum_i (s_i - \bar{s})^2}$$

The cell repeats the example from the slides on the imbalanced loan book, with average precision as the metric. If two models differ by less than their standard deviation, the data cannot tell them apart.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for name, model in [("logistic regression", LogisticRegression(max_iter=1000)),
                    ("decision tree, depth 4", DecisionTreeClassifier(max_depth=4, random_state=SEED))]:
    scores = cross_val_score(model, Xg_train, yg_train, cv=cv, scoring="average_precision")
    print(f"{name:<24} folds {np.round(scores, 3)}   AP = {scores.mean():.2f} ± {scores.std(ddof=1):.2f}")